## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


## Exploratory Data Analysis with Polars 

### Load CSV File into Polars DataFrame

In [3]:
df = pl.read_csv("data/Insurance_Claim_Info_data.csv",
                 ignore_errors=True)

df

Claim Number,City Code,City,Enterprise Type,Claim Type,Claim Site,Product Insured
str,str,str,str,str,str,str
"""DQW1NZO0PL""","""NSK""","""Nashik""","""Public Limited Company""","""Property Loss""","""In Transit""","""Inventory - Raw Material"""
"""JS5GAPRN5B""","""BOM""","""Mumbai""","""One Person Company""","""Property Loss""","""In Transit""","""Cameras and other Misc. Securi…"
"""ZTSVAQSEAQ""","""LKO""","""Lucknow""","""Public Limited Company""","""Property Loss""","""In Transit""","""Fixtures"""
"""EW7NWHI7LI""","""DEL""","""Delhi""","""Sole Proprietorship""","""Property Loss""","""In Transit""","""Pumps and Motors"""
"""UJOFDC41EL""","""DEL""","""Delhi""","""One Person Company""","""Property Loss""","""In Transit""","""Misc. Engineering Tools"""
…,…,…,…,…,…,…
"""5CFGWQ6IR5""","""AGR""","""Agra""","""Public Limited Company""","""Property Loss""","""In Transit""","""Misc. Engineering Tools"""
"""QQ6EAWA4Q5""","""LKO""","""Lucknow""","""Partnership Firm""","""Property Damage""","""In Transit""","""Misc. Electronic Items"""
"""X1J58PT1J5""","""HYD""","""Hyderabad""","""One Person Company""","""Property Damage""","""In Transit""","""Misc. Electronic Items"""


In [4]:
column_data_types = zip(df.columns, df.dtypes)

for column, dtype in column_data_types:
    print(f"{column}:\t\t{dtype}")

Claim Number:		String
City Code:		String
City:		String
Enterprise Type:		String
Claim Type:		String
Claim Site:		String
Product Insured:		String


### Retrieve Basic Information About DataFrame

In [5]:
print(df.shape)
print(df.dtypes)

(34110, 7)
[String, String, String, String, String, String, String]


### Display Summary Statistics for All Columns

In [6]:
summary = df.describe()
print(summary)

shape: (9, 8)
┌────────────┬────────────┬───────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ Claim      ┆ City Code ┆ City       ┆ Enterpris ┆ Claim     ┆ Claim     ┆ Product   │
│ ---        ┆ Number     ┆ ---       ┆ ---        ┆ e Type    ┆ Type      ┆ Site      ┆ Insured   │
│ str        ┆ ---        ┆ str       ┆ str        ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str        ┆           ┆            ┆ str       ┆ str       ┆ str       ┆ str       │
╞════════════╪════════════╪═══════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 34110      ┆ 34110     ┆ 34110      ┆ 34110     ┆ 34110     ┆ 34110     ┆ 34110     │
│ null_count ┆ 0          ┆ 0         ┆ 0          ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
│ mean       ┆ null       ┆ null      ┆ null       ┆ null      ┆ null      ┆ null      ┆ null      │
│ std        ┆ null       ┆ null      ┆ null       ┆ null      ┆ null      ┆ 

### Count of Nulls In Each Feature

In [7]:
for col in df.columns:
    print(f"{col} : {df[col].is_null().sum()}")

Claim Number : 0
City Code : 0
City : 0
Enterprise Type : 0
Claim Type : 0
Claim Site : 0
Product Insured : 0


### Find Longest Text Length in Each Column

In [9]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

Claim Number,City Code,City,Enterprise Type,Claim Type,Claim Site,Product Insured
u32,u32,u32,u32,u32,u32,u32
10,3,13,36,15,10,56


### Count Unique Values in Each Column

In [11]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                 Unique values in Claim Number : 34110 
                    Unique values in City Code : 23    
                         Unique values in City : 23    
              Unique values in Enterprise Type : 13    
                   Unique values in Claim Type : 2     
                   Unique values in Claim Site : 3     
              Unique values in Product Insured : 13    


### Retrieve Unique Values in Column If There Are Fewer Than 30

In [14]:


for col in df.columns:
    unique_values = df[col].unique()
    if len(unique_values) < 30:
        print(f"'{col}' {len(unique_values)}: {unique_values.to_list()}")

'City Code' 23: ['PAT', 'CJB', 'KNU', 'BDQ', 'AMD', 'JAI', 'PNQ', 'BHO', 'BLR', 'STV', 'MAA', 'NSK', 'VSK', 'NGP', 'HYD', 'THN', 'IDR', 'DEL', 'CCU', 'ATQ', 'BOM', 'AGR', 'LKO']
'City' 23: ['Kanpur', 'Jaipur', 'Nagpur', 'Mumbai', 'Bangalore', 'Chennai', 'Kolkata', 'Ahmedabad', 'Nashik', 'Vadodara', 'Coimbatore', 'Lucknow', 'Indore', 'Thane', 'Surat', 'Amritsar', 'Delhi', 'Visakhapatnam', 'Hyderabad', 'Agra', 'Patna', 'Pune', 'Bhopal']
'Enterprise Type' 13: ['Cooperative Society', 'Foreign Subsidary', 'One Person Company', 'Non-Profit Organization (NPO)', 'Partnership Firm', 'Private Limited Company', 'Joint-Venture Company', 'Limited Liability Parterneship (LLP)', 'Public Limited Company', 'Private Ltd. MSME - Small', 'Private Ltd. MSME - Medium', 'Sole Proprietorship', 'Private Ltd. MSME - Micro']
'Claim Type' 2: ['Property Damage', 'Property Loss']
'Claim Site' 3: ['Other', 'In Transit', 'Warehouse']
'Product Insured' 13: ['Misc. Sensors', 'Misc. Electronic Items', 'Misc. Lab Equipme

### Check Distribution of Numerical Columns

In [15]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['id']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')